# Notebook 03 — The Producer: the journey of a message, and every setting that matters

**Goal:** you can explain, step by step, what happens between `producer.produce(...)` in Python and a message sitting in a partition file; and you know what each important producer setting does, its default, and when to change it. Every claim is backed by an experiment you run.

**Needs:** Notebook 01 (Kafka running) and Notebook 02 (topics, partitions, offsets).

| Part | Steps |
|---|---|
| **A. The journey** | 0 Connect · 1 First message · 2 Async nature · 3 Delivery reports · 4 Keys and partitions · 5 When things go wrong |
| **B. The settings** | 6 Identity · 7 Durability (`acks`) · 8 Duplicates, retries, ordering · 9 Throughput (batching, compression) · 10 Size limits and buffers · 11 Ready-made profiles |
| **C. Wrap-up** | 12 Corrections to `config/PRODUCER_CONFIG_GUIDE.md` · 13 Cheat sheet · Checkpoint |

> **Which library, which names?** We use `confluent-kafka`, a Python wrapper over a C library called **librdkafka**. Setting names mostly match the Java client, but **defaults differ** (for example, `enable.idempotence` is off by default here). Every default quoted below comes from librdkafka's official configuration table.

# Part A — The journey of one message

```text
  YOUR PYTHON CODE                       INSIDE THE PRODUCER OBJECT (librdkafka)                    KAFKA BROKER
  ----------------                       ----------------------------------------                   ------------
  producer.produce(topic, key, value)
        |                                
        v   (returns immediately)        [1] Validate: size ok? queue not full?   -> error raised to YOU
                                         [2] Choose PARTITION  (key hash, or sticky if no key)
                                         [3] Put message into the BATCH for that partition (in memory)
                                                   |
                                                   | batch leaves when it is full  (batch.size / batch.num.messages)
                                                   |           or has waited       (linger.ms)
                                                   v
                                         [4] Compress the batch  (compression.type)
                                         [5] Background thread sends it  ----------------------->  [6] LEADER of that partition
                                                                                                       appends to its .log file
                                                                                                       followers copy it
                                                                                                       sends ACK  (per `acks`)
                                         [7] ACK arrives  <---------------------------------------
                                         [8] Delivery report queued
  producer.poll() / flush()  <---------  [9] YOUR callback runs: "stored at partition P, offset O" or "failed: reason"
```

Three facts that explain most producer behavior:
1. `produce()` **does not send anything.** It only puts the message in a memory queue.
2. Sending happens on a **background thread**, in **batches** (one batch = many messages for one partition).
3. You learn the outcome only **later**, through a callback, and only when you call `poll()` or `flush()`.

## Step 0 — Connect

Same helpers as Notebook 02, plus a few for cleanup and measuring.

In [1]:
import json
import random
import re
import subprocess
import time
from pathlib import Path

from confluent_kafka import KafkaException, Producer
from confluent_kafka.admin import AdminClient, NewTopic

BOOTSTRAP = "localhost:9092"
CONTAINER = "kafka-lab"
DATA_DIR = "/var/lib/kafka/data"


def run_on_host(command, show=True):
    """Run a command on your real machine; print its output if show=True; return the process."""
    in_flatpak_sandbox = Path("/.flatpak-info").exists()
    full_command = ["flatpak-spawn", "--host", *command] if in_flatpak_sandbox else command
    result = subprocess.run(full_command, capture_output=True, text=True)
    output = (result.stdout + result.stderr).strip()
    if output and show:
        print(output)
    return result


def kafka_tool(tool, *args, show=True):
    """Run a Kafka command-line tool inside the container."""
    return run_on_host(["docker", "exec", CONTAINER, f"/opt/kafka/bin/{tool}", *args], show=show)


def in_container(shell_command, show=True):
    """Run a shell command inside the container."""
    return run_on_host(["docker", "exec", CONTAINER, "sh", "-c", shell_command], show=show)


admin = AdminClient({"bootstrap.servers": BOOTSTRAP})


def delete_topic(name):
    """Delete a topic if it exists and wait until it is gone."""
    if name in admin.list_topics(timeout=10).topics:
        admin.delete_topics([name])[name].result()
        while name in admin.list_topics(timeout=10).topics:
            time.sleep(0.5)


def recreate_topic(name, partitions=3):
    """Delete the topic if it exists, then create it fresh with the given partition count."""
    delete_topic(name)
    admin.create_topics([NewTopic(name, num_partitions=partitions, replication_factor=1)])[name].result()
    print(f"Created topic '{name}' with {partitions} partition(s)")


def bytes_on_disk(topic):
    """Total size of the message files (.log) of a single-partition topic."""
    out = in_container(f"cat {DATA_DIR}/{topic}-0/*.log | wc -c", show=False).stdout.strip()
    return int(out)


def batch_count(topic):
    """Number of batches stored in a single-partition topic (read from the log files)."""
    out = in_container(
        f"for f in {DATA_DIR}/{topic}-0/*.log; do /opt/kafka/bin/kafka-dump-log.sh --files $f; done | grep -c baseOffset",
        show=False,
    ).stdout.strip()
    return int(out)


print("Connected. Brokers:", [f"id={b.id}" for b in admin.list_topics(timeout=10).brokers.values()])

Connected. Brokers: ['id=1']


## Step 1 — Send the first message

The smallest possible producer. Read each line's comment.

In [4]:
recreate_topic("nb3_first", partitions=3)

# A Producer is configured with a dictionary of settings. "bootstrap.servers" is the only required one.
producer = Producer({"bootstrap.servers": BOOTSTRAP})

# produce(): queue ONE message. Arguments:
#   topic - where to write it
#   value - the message body; bytes (or str, which Python encodes to bytes for you)
producer.produce("nb3_first", value="hello kafka")

print(len(producer))

# flush(): block until every queued message has been sent AND confirmed (or has failed).
# Without it, a short script could exit before anything is actually sent.
remaining = producer.flush()
print("messages still waiting after flush:", remaining)

print(len(producer))

Created topic 'nb3_first' with 3 partition(s)
1
messages still waiting after flush: 0
0


Check that it is really in Kafka. We read the end offsets (the Notebook 02 way):

In [5]:
kafka_tool("kafka-get-offsets.sh", "--bootstrap-server", "localhost:9092", "--topic", "nb3_first", "--time", "latest")

nb3_first:0:0
nb3_first:1:1
nb3_first:2:0


CompletedProcess(args=['flatpak-spawn', '--host', 'docker', 'exec', 'kafka-lab', '/opt/kafka/bin/kafka-get-offsets.sh', '--bootstrap-server', 'localhost:9092', '--topic', 'nb3_first', '--time', 'latest'], returncode=0, stdout='nb3_first:0:0\nnb3_first:1:1\nnb3_first:2:0\n', stderr='')

Expected: the three partitions show end offsets `0, 0, 1` in some order: the message landed in exactly one partition. We did not choose which; **Step 4** explains how Kafka decided.

**What `produce()` accepts** (all optional except `topic`):

| Argument | Meaning |
|---|---|
| `value` | The message body. |
| `key` | Used to choose the partition and to keep related messages in order (Step 4). |
| `partition` | Force a specific partition (rarely a good idea in real systems). |
| `on_delivery` | Function called when the result is known (Step 3). |
| `headers` | Optional list of `(name, bytes)` pairs: metadata such as a trace ID. |
| `timestamp` | Override the message time in milliseconds. Default: now. |

## Step 2 — `produce()` is asynchronous

We send 10,000 messages and time two things: how long the `produce()` calls take, and how long `flush()` takes afterwards.

In [6]:
recreate_topic("nb3_async", partitions=1)
producer = Producer({"bootstrap.servers": BOOTSTRAP})

start = time.time()
for number in range(10_000):
    producer.produce("nb3_async", value=b"x" * 100)
produce_seconds = time.time() - start
print(f"10,000 produce() calls returned in {produce_seconds:.3f} s")
print(f"messages sitting in the producer's memory queue: {len(producer)}")

start = time.time()
producer.flush()
print(f"flush() took {time.time() - start:.3f} s")
print(f"messages left in the queue: {len(producer)}")

Created topic 'nb3_async' with 1 partition(s)
10,000 produce() calls returned in 0.039 s
messages sitting in the producer's memory queue: 10000
flush() took 0.193 s
messages left in the queue: 0


**What you see:** the 10,000 calls finish almost instantly, with all 10,000 messages still queued in memory. The real work (batching, network, broker confirmation) happens afterwards, during `flush()`.

**Why this design?** Sending each message individually and waiting for the answer would limit you to roughly 1 / (network round-trip) messages per second. Queuing and batching lets one network request carry thousands.

**The danger:** if your program crashes or exits while messages are queued, **they are lost**. Always `flush()` before exit.

## Step 3 — Delivery reports (callbacks)

Because sending is asynchronous, Kafka reports each message's fate through a **callback**: a function you provide, which the library calls later with `(error, message)`.

**Important rule:** callbacks run only inside `poll()` or `flush()`, on *your* thread. If you never call either, you never hear back.

In [7]:
recreate_topic("nb3_callbacks", partitions=3)
producer = Producer({"bootstrap.servers": BOOTSTRAP})


def on_delivery(error, message):
    """Called once per message: error is None on success, otherwise a KafkaError."""
    if error is not None:
        print(f"FAILED  {message.value()!r}: {error}")
    else:
        print(f"STORED  {message.value()!r} -> topic={message.topic()} partition={message.partition()} offset={message.offset()}")


for number in range(5):
    producer.produce("nb3_callbacks", value=f"event-{number}", on_delivery=on_delivery)

print("Nothing printed yet: no poll() or flush() so far.\n")
producer.flush()

Created topic 'nb3_callbacks' with 3 partition(s)
Nothing printed yet: no poll() or flush() so far.

STORED  b'event-1' -> topic=nb3_callbacks partition=0 offset=0
STORED  b'event-3' -> topic=nb3_callbacks partition=0 offset=1
STORED  b'event-4' -> topic=nb3_callbacks partition=0 offset=2
STORED  b'event-0' -> topic=nb3_callbacks partition=2 offset=0
STORED  b'event-2' -> topic=nb3_callbacks partition=2 offset=1


0

`message` offers: `.topic()`, `.partition()`, `.offset()`, `.key()`, `.value()`, `.timestamp()`, `.headers()`.

| Pattern | When to use |
|---|---|
| `produce()` in a loop, `poll(0)` occasionally, `flush()` at the end | High-volume sending (the normal way) |
| `produce()` then `flush()` after *every* message | Only when you truly need to wait for each result (slow: one network round trip per message) |
| No callback | "Fire and forget": you never learn about failures |

## Step 4 — Keys decide partitions

Recall: order is guaranteed only **inside one partition**. So if "all events of customer 7 must stay in order", they must all land in the same partition. That is the job of the **key**.

```text
   partition = hash(key) mod number_of_partitions
```
Same key, same hash, same partition, every time.

In [8]:
recreate_topic("nb3_keys", partitions=3)

stored = []     # an ordinary list; the callback adds one (key, partition) pair per message


def remember_partition(error, message):
    """Callback: note which partition Kafka stored this message in."""
    key = message.key().decode()                # the key arrives as bytes; turn it back into text
    stored.append((key, message.partition()))


for attempt in range(2):            # two completely separate producers, to prove it is not luck
    producer = Producer({"bootstrap.servers": BOOTSTRAP})
    for number in range(12):
        producer.produce("nb3_keys", key=f"customer-{number % 4}", value="event", on_delivery=remember_partition)
    producer.flush()

print(f"{len(stored)} messages were stored. Distinct (key, partition) pairs:")
for key, partition in sorted(set(stored)):      # set() removes repeated pairs
    print(f"  {key} -> partition {partition}")

Created topic 'nb3_keys' with 3 partition(s)
24 messages were stored. Distinct (key, partition) pairs:
  customer-0 -> partition 1
  customer-1 -> partition 0
  customer-2 -> partition 0
  customer-3 -> partition 2


24 messages were sent, but only **4 lines** are printed, one per customer. If a customer had ever landed in two partitions, it would appear on two lines. It never does: each key always goes to the same partition, even across two separate producers. Several customers may share a partition; that is fine.

**Messages without a key.** Kafka spreads them out, but in a surprising way:

In [9]:
chosen = []
producer = Producer({"bootstrap.servers": BOOTSTRAP})
for number in range(12):
    producer.produce("nb3_keys", value="no key", on_delivery=lambda error, message: chosen.append(message.partition()))
producer.flush()
print("12 messages, sent together, no key:", chosen)

12 messages, sent together, no key: [1, 1, 1, 1, 2, 0, 0, 0, 0, 0, 0, 0]


Usually you see long runs of the same partition. This is the **sticky partitioner**: it fills one partition's batch, then moves to another. Reason: one full batch to one partition is much cheaper than many tiny batches to different partitions. Over time the load still evens out.

**Rules of thumb**

| Situation | Do this |
|---|---|
| Need ordering per entity (customer, device, account) | Use that entity's ID as the key |
| No ordering needed | No key (best spread and batching) |
| One key is extremely busy (a "hot key") | That partition becomes a bottleneck; consider a more specific key |

**Gotchas**
- **Changing the partition count later changes `hash(key) mod N`**, so the same key may move partitions, and per-key ordering across the change is lost. Choose partition counts with growth in mind.
- **Different client libraries hash differently.** The Java client uses *murmur2*; librdkafka's default (`partitioner = consistent_random`) uses CRC32. If Java and Python producers write to the same topic and must agree on partitions, set `"partitioner": "murmur2_random"` in Python.

## Step 5 — When things go wrong

Failures reach you in **two different ways**. Knowing which is which matters.

| Kind | How you learn | Example |
|---|---|---|
| **Immediate** | `produce()` raises an exception | Message too large; local queue full |
| **Later** | Your callback receives an `error` | Broker unreachable; delivery timed out |

### 5a — Immediate: message too big

In [10]:
producer = Producer({"bootstrap.servers": BOOTSTRAP})
try:
    producer.produce("nb3_keys", value=b"x" * 2_000_000)      # 2 MB; the default limit is about 1 MB
except KafkaException as problem:
    print("Raised immediately:", problem)

Raised immediately: KafkaError{code=MSG_SIZE_TOO_LARGE,val=10,str="Unable to produce message: Broker: Message size too large"}


### 5b — Later: broker unreachable

We point a producer at a port where nothing listens, and give up after 3 seconds (`message.timeout.ms`). `log_level: 0` hides the library's own connection-error chatter so you can see just our result.

In [11]:
outcome = []
lost_producer = Producer({"bootstrap.servers": "localhost:9999", "message.timeout.ms": 3000, "log_level": 0})
lost_producer.produce("nb3_keys", value="will never arrive", on_delivery=lambda error, message: outcome.append(error))

start = time.time()
lost_producer.flush()
print(f"After {time.time() - start:.1f} s the callback received: {outcome[0]}")

After 3.0 s the callback received: KafkaError{code=_MSG_TIMED_OUT,val=-192,str="Local: Message timed out"}


`_MSG_TIMED_OUT` means: "the producer tried for the whole time budget and gave up". This is the error you will see in real outages. Because it arrives through the callback, **if you set no callback, this message is silently lost.**

### 5c — The surprise: a typo creates a new topic

In [12]:
outcome = []
producer = Producer({"bootstrap.servers": BOOTSTRAP, "message.timeout.ms": 5000})
producer.produce("nb3_typo_ordrs", value="oops", on_delivery=lambda error, message: outcome.append(error))
producer.flush()
print("Callback error:", outcome[0])
print("Topic now exists:", "nb3_typo_ordrs" in admin.list_topics(timeout=10).topics)

Callback error: None
Topic now exists: True


No error: Kafka silently **created** the misspelled topic. Our lab broker has `auto.create.topics.enable` at its default (**true**), so the first write to an unknown name creates it. In production that is usually turned off (`false`), so typos fail loudly instead of creating junk topics.

In [13]:
delete_topic("nb3_typo_ordrs")
print("Cleaned up.")

Cleaned up.


%5|1792053126.373|PARTCNT|rdkafka#producer-12| [thrd:main]: Topic nb3_typo_ordrs (id ZbMpjJkYRqKabjvE1FNh2Q) partition count changed from 3 to 0


---
# Part B — The settings

For each setting: **what it does · default · why it exists · when to change it.** Defaults are librdkafka's. Where an experiment is possible, you run it.

## Step 6 — Identity and connection

| Setting | Default | Meaning | Advice |
|---|---|---|---|
| `bootstrap.servers` | *(none, required)* | Comma-separated list of `host:port` front doors. Used only to **discover** the cluster: the client connects to one, asks "who is in the cluster?", then talks to all brokers directly. | List 2-3 brokers (not one) so startup works if one is down. You do not need to list them all. |
| `client.id` | `rdkafka` | A name sent with every request. It does not change behavior. It appears in broker logs, metrics and quotas. | Always set a meaningful one, like `"order-service-producer"`. During an incident, "who is sending this traffic?" has an answer. |

Not covered here, but you will meet them in production: `security.protocol`, `sasl.*`, `ssl.*` (encryption and login).

The first connection takes a moment; that's the discovery step. You can see what the client learned:

In [14]:
producer = Producer({"bootstrap.servers": BOOTSTRAP, "client.id": "notebook-03-demo"})
cluster = producer.list_topics(timeout=10)
print("Client discovered brokers:", [f"{b.host}:{b.port}" for b in cluster.brokers.values()])

Client discovered brokers: ['localhost:9092']


## Step 7 — Durability: `acks`

`acks` answers: **"How many copies of my message must exist before Kafka says 'done'?"**

```text
   producer ---> LEADER (writes to its log) ---> follower 1 (copies)
                                            \--> follower 2 (copies)

   acks=0     : producer does not wait for anything.                  fastest, can lose data silently
   acks=1     : wait for the LEADER's write only.                     lost if the leader dies before followers copy
   acks=all   : wait until ALL in-sync replicas (ISR) have it.        safest
```

| Value | Waits for | Risk | Typical use |
|---|---|---|---|
| `0` | Nothing | Message lost even if the broker is down; you are never told | Metrics/logs where loss is acceptable |
| `1` | Leader only | Lost if the leader crashes right after answering | Rarely the right choice today |
| `all` (or `-1`) | Every in-sync replica | Slower; blocked if too few replicas are in sync (see below) | **Default in librdkafka. Use for anything that matters.** |

**`acks=all` and `min.insync.replicas` work as a pair.** "All in-sync replicas" could be just one if the others fell behind! `min.insync.replicas=2` (a topic/broker setting) says "refuse writes unless at least 2 copies are in sync". Together, with replication factor 3: you survive one broker failure with no data loss and no downtime. Our lab has one broker, so we cannot demonstrate this. Remember the recipe: **`replication.factor=3`, `min.insync.replicas=2`, `acks=all`.**

### Experiment: what does waiting cost?
We send 50 messages one at a time, waiting for each result, under each setting.

In [15]:
recreate_topic("nb3_acks", partitions=1)

for acks in ["0", "1", "all"]:
    offsets_seen = []
    producer = Producer({"bootstrap.servers": BOOTSTRAP, "acks": acks, "linger.ms": 0})
    producer.produce("nb3_acks", value="warm-up")      # first send pays for connecting; keep it out of the timing
    producer.flush()
    start = time.time()
    for number in range(50):
        producer.produce("nb3_acks", value="x", on_delivery=lambda error, message: offsets_seen.append(message.offset()))
        producer.flush()
    average_ms = (time.time() - start) / 50 * 1000
    print(f"acks={acks:<3}  average {average_ms:5.2f} ms per message   offset reported to callback: {offsets_seen[-1]}")

Created topic 'nb3_acks' with 1 partition(s)
acks=0    average  0.14 ms per message   offset reported to callback: None
acks=1    average  1.58 ms per message   offset reported to callback: 101
acks=all  average  1.49 ms per message   offset reported to callback: 152


**What to notice**
- `acks=0` should be the fastest (it never waits for the broker), and the callback gets **no offset** (`None`): Kafka never answered, so the producer cannot know where, or whether, the message was stored.
- `acks=1` and `acks=all` look the same *here* because there is only one broker. With 3 brokers, `all` waits for the slowest in-sync follower, which costs a few extra milliseconds.

## Step 8 — Duplicates, retries and ordering

Networks fail. Consider this sequence:

```text
  producer ---- message M ---->  broker writes M  (offset 7)
  producer <-- ACK lost ------   (network drops the answer)
  producer: "no answer... send M again"  ---->  broker writes M AGAIN (offset 8)     <- DUPLICATE
```
Retrying is necessary (otherwise failures lose data) but, naively, retrying creates duplicates. **Idempotence** fixes this: each producer gets an ID, and each message a sequence number, so the broker can recognize "I already have sequence 5 from producer 12" and ignore the repeat.

### The settings

| Setting | Default | Meaning |
|---|---|---|
| `enable.idempotence` | **`false`** in librdkafka (the Java client defaults to `true`) | Turns on producer-ID + sequence numbers: no duplicates from retries, and order is preserved within a partition. |
| `retries` | `2147483647` (practically unlimited) | How many times to resend a failed batch. |
| `retry.backoff.ms` | `100` | Wait before the first retry; grows (doubling) up to `retry.backoff.max.ms` (default `1000`). Stops a struggling broker being hammered. |
| `message.timeout.ms` (alias `delivery.timeout.ms`) | `300000` (5 min) | **The real limit on retrying**: total time allowed from `produce()` to final result, retries included. After this, the callback gets `_MSG_TIMED_OUT`. |
| `request.timeout.ms` | `30000` | How long the producer waits for the broker's answer to *one* request before treating it as failed. |
| `max.in.flight.requests.per.connection` | `1000000` | How many sent-but-unanswered requests are allowed per broker connection. |

So the practical question about retries is not "how many?" but **"how long am I willing to keep trying?"**, which is `message.timeout.ms`.

### What turning on idempotence changes
The library forces safe values and **refuses to start** if you ask for something contradictory. Try it:

In [16]:
conflicts = [
    {"enable.idempotence": True, "acks": "0"},
    {"enable.idempotence": True, "max.in.flight.requests.per.connection": 6},
    {"enable.idempotence": True, "retries": 0},
]
for extra in conflicts:
    try:
        Producer({"bootstrap.servers": BOOTSTRAP, "log_level": 0, **extra})
        print(extra, "-> accepted")
    except KafkaException as problem:
        print(f"{extra}\n    -> REFUSED: {problem.args[0].str()}")

{'enable.idempotence': True, 'acks': '0'}
    -> REFUSED: Failed to create producer: `acks` must be set to `all` when `enable.idempotence` is true
{'enable.idempotence': True, 'max.in.flight.requests.per.connection': 6}
    -> REFUSED: Failed to create producer: `max.in.flight` must be set <= 5 when `enable.idempotence` is true
{'enable.idempotence': True, 'retries': 0}
    -> REFUSED: Failed to create producer: `retries` must be set >= 1 when `enable.idempotence` is true


So idempotence implies: `acks=all`, `retries >= 1`, `max.in.flight <= 5`.

### See the fingerprint on disk

First, one idea. Kafka does not store messages one by one. Messages sent together are packed into a **batch**, like items in a parcel. Every parcel has a **label on the outside** describing what is inside. That label is called the **batch header**: it is Kafka's own bookkeeping, separate from your data.

```text
  BATCH (one parcel on disk)
  +-----------------------------------------------+
  | LABEL (batch header):                         |
  |    baseOffset   = 3   (first message is #3)   |
  |    count        = 3   (3 messages inside)     |
  |    producerId   = 1   (who sent it)           |  <- the fingerprint
  |    baseSequence = 0   (sequence numbers start)|
  +-----------------------------------------------+
  | message 3: "idempotent"                       |
  | message 4: "idempotent"                       |  <- your data
  | message 5: "idempotent"                       |
  +-----------------------------------------------+
```

The two fingerprint boxes are `producerId` and `baseSequence`. With idempotence **off** they hold `-1`, meaning "not used". With it **on** they hold real numbers; the broker uses them to spot repeated messages.

The experiment: send 3 messages with idempotence off, then 3 with it on, and print each batch's label.


In [ ]:
recreate_topic("nb3_idempotence", partitions=1)

for label, idempotent in [("plain", False), ("idempotent", True)]:
    producer = Producer({"bootstrap.servers": BOOTSTRAP, "enable.idempotence": idempotent, "log_level": 0})
    for number in range(3):
        producer.produce("nb3_idempotence", value=label)
    producer.flush()

# Ask Kafka's log-reading tool to print everything in the log file.
log_file = f"{DATA_DIR}/nb3_idempotence-0/00000000000000000000.log"
dump = in_container(f"/opt/kafka/bin/kafka-dump-log.sh --files {log_file}", show=False).stdout

# Each batch label is a line that starts with "baseOffset". Print only the four boxes we care about.
for line in dump.splitlines():
    if line.startswith("baseOffset"):
        words = line.split()                                  # split the line into words
        label = {name: words[position + 1] for position, name in enumerate(words) if name in
                 ("baseOffset:", "count:", "producerId:", "baseSequence:")}
        print(label)

**Look at the two labels.**

| | `producerId` | `baseSequence` | Meaning |
|---|---|---|---|
| First batch (idempotence **off**) | `-1` | `-1` | Nothing to check. If this batch were sent twice, Kafka would store it twice. |
| Second batch (idempotence **on**) | a real number | `0` | Every message from this producer is numbered. If "producer 1, sequence 2" arrives again, the broker says "already have it" and ignores it. |

### Ordering and `max.in.flight`
If several requests are in flight and the *first* fails and is retried after the second succeeded, messages end up in the wrong order. Two ways to be safe:
- `max.in.flight = 1`: only one request at a time (safe, but slow); **or**
- `enable.idempotence = true`: the broker uses sequence numbers to keep order, and **up to 5** requests in flight is safe.

The second option is the modern answer. Idempotence gives you both safety and speed.

> **Scope of the guarantee.** Idempotence prevents duplicates caused by *producer retries* within one producer session and one partition. It does not remove duplicates caused by your *own* code (for example, re-reading the same source record and producing it twice), nor does it by itself give end-to-end "exactly-once" (that needs transactions, a later topic).

## Step 9 — Throughput: batching and compression

Recall the journey: messages wait in a per-partition **batch** until it is full or old enough. Three settings control this.

| Setting | Default | Meaning | Trade-off |
|---|---|---|---|
| `linger.ms` (alias `queue.buffering.max.ms`) | **`5`** | Max time a batch waits for more messages before being sent anyway. | Higher = bigger batches and better throughput, but each message may wait longer. `0` = send immediately. |
| `batch.size` | `1000000` (1 MB) | Max bytes in one batch. | Bigger = fewer requests. Memory is used per active partition. |
| `batch.num.messages` | `10000` | Max messages in one batch. | A batch is sent when **either** this or `batch.size` is reached. |
| `compression.type` | `none` | Compress each batch: `gzip`, `snappy`, `lz4`, `zstd`. | Saves network and disk; costs CPU. Larger batches compress better. |

> The Java client's default `batch.size` is only 16 KB, so numbers you read in Java-focused articles will not match this library.

### Experiment 1: `linger.ms`
We send the same 20,000 JSON messages (about 220 bytes each) three ways and then count how many **batches** reached the log.

In [ ]:
random.seed(1)
words = ["alpha", "bravo", "charlie", "delta", "echo", "foxtrot", "golf", "hotel"]
sample_messages = [
    json.dumps({
        "order_id": number,
        "customer": f"customer-{number % 100}",
        "status": "PLACED",
        "items": [{"sku": f"SKU-{random.randint(1, 50)}", "qty": random.randint(1, 5)} for _ in range(3)],
        "note": " ".join(random.choices(words, k=8)),
    }).encode()
    for number in range(20_000)
]
print(f"{len(sample_messages):,} messages, average {sum(map(len, sample_messages)) // len(sample_messages)} bytes each")


def send_all(topic, extra_settings):
    """Send every sample message to a fresh one-partition topic; return (seconds, bytes_on_disk, batches)."""
    recreate_topic(topic, partitions=1)
    producer = Producer({"bootstrap.servers": BOOTSTRAP, "log_level": 0, **extra_settings})
    start = time.time()
    for message in sample_messages:
        try:
            producer.produce(topic, value=message)
        except BufferError:               # local queue full: let it drain, then retry
            producer.poll(0.5)
            producer.produce(topic, value=message)
        producer.poll(0)
    producer.flush()
    return time.time() - start, bytes_on_disk(topic), batch_count(topic)

In [ ]:
print(f"{'linger.ms':>10} {'seconds':>9} {'batches':>9}")
for linger in [0, 5, 50]:
    seconds, size, batches = send_all(f"nb3_linger_{linger}", {"linger.ms": linger})
    print(f"{linger:>10} {seconds:>9.2f} {batches:>9}")

**What to notice:** with `linger.ms=0` the producer sends whatever is queued the instant the network is free, producing hundreds of small batches. Allowing just 5 ms to collect messages cuts it to a handful of large batches and is several times faster. Going from 5 to 50 adds little here, because the batches were already big. Your exact numbers will vary.

### Experiment 2: compression
Same messages, same `linger.ms`, different codecs. We compare the size of the message files on disk.

In [ ]:
results = {}
for codec in ["none", "snappy", "lz4", "zstd", "gzip"]:
    seconds, size, batches = send_all(f"nb3_codec_{codec}", {"compression.type": codec})
    results[codec] = (seconds, size)

baseline = results["none"][1]
print(f"{'codec':>8} {'seconds':>9} {'bytes on disk':>15} {'% of original':>15}")
for codec, (seconds, size) in results.items():
    print(f"{codec:>8} {seconds:>9.2f} {size:>15,} {size / baseline * 100:>14.0f}%")

**Reading it**
- The data is repetitive JSON, so compression is very effective here; real data compresses less. Treat the *ranking*, not the exact percentages, as the lesson.
- Typical character of each codec:

| Codec | Speed | Compression | Notes |
|---|---|---|---|
| `none` | n/a | none | Fine on a fast local network with small messages |
| `lz4` | very fast | moderate | Good default when CPU is precious |
| `snappy` | very fast | moderate | Similar to lz4 |
| `zstd` | fast | **high** | Usually the best all-round choice today |
| `gzip` | slowest | highest or near it | Heavy on CPU; use when bandwidth is very expensive |

- Compression happens **per batch**. That is another reason `linger.ms > 0` helps: bigger batches compress better.
- The broker stores the batch as compressed (the topic setting `compression.type=producer` means "keep what the producer sent"). Consumers decompress automatically.

## Step 10 — Size limits and buffers

| Setting | Default | Meaning |
|---|---|---|
| `message.max.bytes` | `1000000` | The largest request the **client** will send. A single message bigger than this is rejected locally (Step 5a). |
| *topic* `max.message.bytes` / *broker* `message.max.bytes` | about `1048588` (1 MiB + header) | The largest batch the **broker** will accept. If the broker's limit is smaller than your client's, the broker refuses the message. Raise **both sides** together if you truly need larger messages. |
| `queue.buffering.max.messages` | `100000` | How many messages may wait in the producer's memory queue. When full, `produce()` raises `BufferError`. |
| `queue.buffering.max.kbytes` | `1048576` (1 GB) | The same limit in bytes; whichever limit is hit first applies. |

Kafka is built for many small messages. Multi-megabyte messages strain everything: memory, replication, consumers. If you have large payloads (images, documents), store them elsewhere (object storage) and send **a reference** (a URL or ID) through Kafka.

### What a full queue looks like
We make a producer that cannot deliver (wrong port) and allow only 5 queued messages.

In [ ]:
stuck_producer = Producer({"bootstrap.servers": "localhost:9999", "queue.buffering.max.messages": 5, "log_level": 0})
try:
    for number in range(10):
        stuck_producer.produce("nb3_keys", value="x")
except BufferError as problem:
    print(f"produce() call number {number + 1} raised BufferError: {problem}")
    print("Normal reaction: call poll() to let the queue drain, wait a moment, and try again.")

A `BufferError` is **back-pressure**: the producer is saying "I produce faster than Kafka accepts". Ignoring it loses messages; handling it (poll, wait, retry) is the standard pattern, as in `send_all` above.

## Step 11 — Ready-made profiles

Three starting points. Copy the one closest to your need, then change only what measurements justify.

```python
SAFE_DEFAULT = {                      # anything that matters: payments, orders, state changes
    "bootstrap.servers": "broker1:9092,broker2:9092,broker3:9092",
    "client.id": "my-service-producer",
    "enable.idempotence": True,       # implies acks=all, retries, max.in.flight <= 5
    "acks": "all",
    "message.timeout.ms": 120_000,    # how long you are willing to keep trying
    "linger.ms": 5,
    "compression.type": "zstd",
}

HIGH_THROUGHPUT = {                   # logs, clickstream, metrics
    **SAFE_DEFAULT,
    "linger.ms": 50,
    "batch.size": 2_000_000,
    "compression.type": "zstd",
}

LOW_LATENCY = {                       # each message should leave immediately
    **SAFE_DEFAULT,
    "linger.ms": 0,
    "compression.type": "none",
}
```
Plus, on the **topic/broker** side for safe profiles: `replication.factor=3`, `min.insync.replicas=2`.

Note what is *not* tuned: threads, socket buffers, and so on. Defaults are good; measure first.

---
# Part C — Wrap-up

## Step 12 — Corrections to `config/PRODUCER_CONFIG_GUIDE.md`

That guide is a decent overview, but you asked me not to trust existing material blindly, so here is what I checked against the library and a live broker:

| The guide says | Reality |
|---|---|
| For ordering with idempotence, "set `max.in.flight.requests.per.connection` to 1" | **Not needed.** With idempotence, up to **5** is safe and ordered (Step 8). `1` just slows you down. |
| "Set `retries` very high" as the main control | True, and it is the default already, but the setting that actually *bounds* retrying is **`message.timeout.ms`** (a.k.a. `delivery.timeout.ms`). |
| `acks=all` = "the leader and all in-sync replicas confirm" | Right, but incomplete: "all in-sync" could mean 1 replica. The guarantee comes from `acks=all` **plus** `min.insync.replicas`. |
| Idempotence is "essential for exactly-once delivery" | It prevents duplicates from *producer retries* only. End-to-end exactly-once also needs transactions and consumer-side care. |
| "`gzip` or `lz4`" for maximum compression | `lz4` is a *speed* codec. For best ratios use **`zstd`** or `gzip` (Step 9). |
| `message.max.bytes` "on the producer side" | True for librdkafka. In the Java client the equivalent is `max.request.size`. Names differ between libraries. |
| Does not mention that defaults differ between clients | librdkafka: `enable.idempotence=false`, `linger.ms=5`, `batch.size=1 MB`. Java: idempotence on by default. Always check *your* library's defaults. |

## Step 13 — Cheat sheet

| Question | Setting |
|---|---|
| Where is Kafka? | `bootstrap.servers` |
| Who am I (for logs and metrics)? | `client.id` |
| How safe is a write? | `acks=all` + topic `min.insync.replicas=2` + `replication.factor=3` |
| No duplicates on retry? | `enable.idempotence=true` |
| How long do I keep retrying? | `message.timeout.ms` |
| Fast vs. low delay? | `linger.ms` (and `batch.size`) |
| Smaller on the wire/disk? | `compression.type=zstd` |
| Which partition? | the **key** (and `partitioner`) |
| Too big? | `message.max.bytes` (client) and `max.message.bytes` (broker/topic) |
| Producing faster than Kafka accepts? | handle `BufferError`; `queue.buffering.max.*` |

**Habits that prevent most incidents**
1. Always give a delivery callback and look at errors.
2. Always `flush()` before exiting.
3. Use a key whenever order per entity matters.
4. Turn `auto.create.topics.enable` off in production.

## Clean up

In [ ]:
created_by_this_notebook = [
    "nb3_first", "nb3_async", "nb3_callbacks", "nb3_keys", "nb3_acks", "nb3_idempotence",
    "nb3_linger_0", "nb3_linger_5", "nb3_linger_50",
    "nb3_codec_none", "nb3_codec_snappy", "nb3_codec_lz4", "nb3_codec_zstd", "nb3_codec_gzip",
]
for topic in created_by_this_notebook:
    delete_topic(topic)
print("Remaining topics:", sorted(t for t in admin.list_topics(timeout=10).topics if not t.startswith("__")))

## Checkpoint — answer without scrolling up

1. `produce()` returned. Is the message in Kafka yet? What call makes sure?
2. When does your delivery callback run?
3. Two customers' events must each stay in order. What do you set on each message, and why does it work?
4. You change a topic from 3 to 6 partitions. What happens to the key-to-partition mapping?
5. What is the difference between an error raised by `produce()` and an error given to the callback? Give one example of each.
6. Describe `acks=0`, `1`, and `all` in one sentence each. Which setting pairs with `acks=all` to make it meaningful?
7. The ack for a message is lost in the network and the producer retries. Without idempotence, what goes wrong? With it, what does the broker do?
8. Which setting limits *how long* the producer keeps retrying?
9. Why does `linger.ms=5` give much higher throughput than `0`, and what does it cost?
10. Why are bigger batches good for compression?

**Next: Notebook 04 — the Consumer.** Reading, offsets and commits, consumer groups, and what happens when a consumer joins or dies (rebalancing).